# Efficient Request Handling: The Scheduler

In our last notebook, we saw how the `EngineCore` acts as the central brain of the vLLM system. Now, we'll zoom in on one of its most critical helpers: the Scheduler. By the end of this notebook, you will understand how the scheduler acts as a brilliant air traffic controller for incoming requests, deciding who gets to "land" on the GPU, when, and with whom, ultimately maximizing the system's throughput and keeping latency low.

In [ ]:
# === Setup Cell ===
# This notebook is self-contained. All necessary imports are here.

import collections
import time
import uuid
from dataclasses import dataclass, field
from typing import List, Dict, Optional

# For visualization
import graphviz

# A simple dataclass to represent a user's request.
# In a real system, this would contain much more information.
@dataclass
class InferenceRequest:
    request_id: str = field(default_factory=lambda: str(uuid.uuid4()))
    prompt: str = ""
    prompt_len: int = 0
    # The state of the request from the scheduler's perspective.
    status: str = "WAITING"

    def __post_init__(self):
        self.prompt_len = len(self.prompt.split()) # A simple word count for length

### The Core Idea: An Air Traffic Controller for GPUs

Imagine a busy airport with only one runway (our expensive GPU). Planes (inference requests) are constantly arriving, all wanting to land as quickly as possible. If we let them land one by one, the runway will be empty between each landing, which is incredibly inefficient. A skilled air traffic controller solves this.

The controller's job is to:
1.  **Queue:** Keep track of all planes waiting in a holding pattern (`waiting_queue`).
2.  **Prioritize:** Decide the order. Maybe a plane with a medical emergency (a high-priority request) gets to land first.
3.  **Batch:** Look at the runway's capacity and weather conditions. Instead of landing one small plane, the controller might group three small planes to land in quick succession (creating a "batch"). This keeps the runway constantly active.
4.  **Manage Resources:** Know exactly when the runway is free to accept the next batch.

The vLLM Scheduler does the exact same thing for LLM inference requests. It takes incoming requests, holds them in a queue, and then intelligently groups them into a *batch* that will be sent to the GPU for processing. Its primary goal is to keep the GPU "runway" as busy as possible to maximize throughput, while also ensuring requests don't wait in the queue for too long.

Let's build a simplified version of this controller.

In [ ]:
class SimpleScheduler:
    """A mock scheduler that mimics the core logic of vLLM's scheduler."""

    def __init__(self, max_tokens_in_batch: int):
        # The "runway capacity" - max total tokens we can handle in one go.
        self.max_tokens_in_batch = max_tokens_in_batch
        # The "holding pattern" for incoming requests (planes).
        self.waiting_queue: collections.deque[InferenceRequest] = collections.deque()
        # A dictionary to quickly find requests that are currently being processed.
        self.running_map: Dict[str, InferenceRequest] = {}

    def add_request(self, request: InferenceRequest):
        """Adds a new request to the waiting queue."""
        print(f"SCHEDULER: 📥 Adding request {request.request_id[:6]} to queue.")
        self.waiting_queue.append(request)

    def schedule(self) -> List[InferenceRequest]:
        """The core logic: create a batch of requests to run on the GPU."""
        print("\nSCHEDULER: 🧐 Scheduling next batch...")
        batch_to_run: List[InferenceRequest] = []
        current_batch_tokens = 0

        # Prioritize requests that are already running (for continuous generation)
        # For simplicity, we'll just re-add all running requests to the next batch.
        # In vLLM, this is more complex, as it handles new tokens.
        batch_to_run.extend(self.running_map.values())
        for req in batch_to_run:
            current_batch_tokens += 1 # Assume each running request generates 1 new token

        # Now, try to add new requests from the waiting queue.
        while self.waiting_queue:
            # Look at the first request in the queue (First-Come, First-Served)
            request = self.waiting_queue[0]
            
            # Can we fit this new request's prompt tokens in our batch?
            if current_batch_tokens + request.prompt_len <= self.max_tokens_in_batch:
                request = self.waiting_queue.popleft() # Take it out of the queue
                request.status = "RUNNING"
                self.running_map[request.request_id] = request
                batch_to_run.append(request)
                current_batch_tokens += request.prompt_len
                print(f"SCHEDULER: ✅ Adding {request.request_id[:6]} to batch.")
            else:
                # If we can't fit the next request, stop building the batch.
                print(f"SCHEDULER: ⏸️ Batch full. Cannot fit {request.request_id[:6]} ({request.prompt_len} tokens).")
                break
        
        if not batch_to_run:
            print("SCHEDULER: 텅 No requests to run.")

        return batch_to_run

    def finish_request(self, request_id: str):
        """Called when a request has finished generation."""
        if request_id in self.running_map:
            request = self.running_map.pop(request_id)
            request.status = "COMPLETED"
            print(f"SCHEDULER: ✅ Request {request.request_id[:6]} finished and removed.")

### Walking Through the `SimpleScheduler`

Let's break down our air traffic controller.

-   **`__init__(self, max_tokens_in_batch)`**: When we create our scheduler, we give it one crucial piece of information: the maximum number of tokens it can pack into a single batch. This is an abstraction for the GPU's memory limit. It also initializes an empty `waiting_queue` (our holding pattern) and a `running_map` to track requests currently on the GPU.

-   **`add_request(self, request)`**: This is simple. When a new request arrives, it's added to the end of the `waiting_queue`. It's now officially in the holding pattern.

-   **`schedule(self)`**: This is the heart of the scheduler. It's called repeatedly by the `EngineCore` to ask, "Who's next?".
    1.  It starts building a new `batch_to_run`.
    2.  First, it automatically includes any requests that are *already running*. In a real LLM, a request isn't finished in one go; it generates one token at a time. So, requests that are still generating must be included in the next batch to get their *next* token.
    3.  Then, it peeks at the front of the `waiting_queue`. It asks a critical question: "If I add this new request to the batch, will I exceed my `max_tokens_in_batch` limit?"
    4.  If the answer is yes, it moves the request from `waiting` to `running`, adds it to the batch, and updates the token count.
    5.  If the answer is no, it stops. The batch is as full as it can be for now.
    6.  Finally, it returns the completed batch, ready to be sent to the Executor.

-   **`finish_request(self, request_id)`**: This is the signal from the Executor that a request is done (e.g., it generated an `<EOS>` token). The scheduler removes it from the `running_map`, freeing up space in future batches for new requests from the queue.

In [ ]:
# === Demonstration ===

# 1. Initialize the Scheduler with a small capacity for demonstration.
# This means we can process a maximum of 30 tokens in a single batch.
scheduler = SimpleScheduler(max_tokens_in_batch=30)

# 2. A few requests arrive.
req1 = InferenceRequest(prompt="The capital of France is") # 5 tokens
req2 = InferenceRequest(prompt="Write a short story about a robot.") # 8 tokens
req3 = InferenceRequest(prompt="Explain the theory of relativity in simple terms for a child.") # 12 tokens
req4 = InferenceRequest(prompt="A recipe for chocolate chip cookies is") # 7 tokens

scheduler.add_request(req1)
scheduler.add_request(req2)
scheduler.add_request(req3)
scheduler.add_request(req4)

print(f"\nInitial waiting queue: {len(scheduler.waiting_queue)} requests")

# 3. The Engine asks the scheduler to create the first batch.
# It should pick req1 (5), req2 (8), and req3 (12). Total tokens = 25.
# It cannot fit req4 (7) because 25 + 7 > 30.
batch1 = scheduler.schedule()
print(f"\n🚀 Batch 1 to be processed ({len(batch1)} requests): {[r.request_id[:6] for r in batch1]}")
print(f"   Waiting queue size after scheduling: {len(scheduler.waiting_queue)}")
print(f"   Running map size after scheduling: {len(scheduler.running_map)}")


# 4. Simulate the Executor finishing one of the requests from the batch.
print("\n...Executor is processing batch 1...")
time.sleep(0.1)
# Let's say req2 finishes first.
scheduler.finish_request(req2.request_id)
# Now, req1 and req3 are still running.

# 5. The Engine asks for the next batch.
# The scheduler will automatically include the still-running req1 and req3.
# Let's assume they each need 1 token for their next generation step (total 2 tokens).
# This leaves 30 - 2 = 28 tokens of space. It can now fit req4 (7 tokens)!
batch2 = scheduler.schedule()
print(f"\n🚀 Batch 2 to be processed ({len(batch2)} requests): {[r.request_id[:6] for r in batch2]}")
print(f"   Waiting queue size after scheduling: {len(scheduler.waiting_queue)}")
print(f"   Running map size after scheduling: {len(scheduler.running_map)}")

### Going Deeper: Scheduling is Memory Management

Our `SimpleScheduler` made a decision based on `max_tokens_in_batch`. This is a good starting point, but it's an abstraction. What does that token limit *really* represent? In vLLM, it represents the available space in the **KV Cache**.

The non-obvious design decision in vLLM is that **scheduling and memory management are two sides of the same coin.** The scheduler doesn't just ask, "Do I have capacity for 500 tokens?". It asks a more specific question to a `BlockSpaceManager`: "I have a new request that needs 10 prompt tokens, and will generate up to 20 new tokens. Do you have 30 free memory blocks to assign to it?"

This has a powerful implication: vLLM can preempt or "swap" requests. Imagine a very long-running request is consuming a lot of KV cache blocks. If a higher-priority request arrives and there's no space, the scheduler can decide to save the state of the long-running request to CPU RAM (swapping it out), free up its KV cache blocks, and let the high-priority request run. Once that's done, it can swap the original request back in.

This tight coupling between the scheduler's decisions and the fine-grained state of the physical memory cache is a key reason vLLM achieves such high throughput. It never allocates more than it needs and can dynamically shuffle requests to keep the GPU fed with optimally packed batches.

In [ ]:
# === Visualization of a Request's Lifecycle ===
# This diagram shows the journey of a single request through the states
# managed by the scheduler.

dot = graphviz.Digraph('RequestLifecycle', comment='The Lifecycle of an Inference Request')
dot.attr('node', shape='box', style='rounded', fontname='Helvetica')
dot.attr(rankdir='LR') # Left to Right layout

# Define the states (nodes)
dot.node('WAITING', 'WAITING\n(In Queue)')
dot.node('RUNNING', 'RUNNING\n(In Batch, on GPU)')
dot.node('COMPLETED', 'COMPLETED\n(Finished)')

# Define the transitions (edges)
dot.edge('WAITING', 'RUNNING', label='scheduler.schedule()')
dot.edge('RUNNING', 'RUNNING', label='Executor generates token')
dot.edge('RUNNING', 'COMPLETED', label='scheduler.finish_request()')

# Add an entry point
dot.node('start', '', shape='point')
dot.edge('start', 'WAITING', label=' engine.add_request() ')


print("Displaying the state machine diagram for a request's lifecycle:")
dot

### Summary: What We Built and Why It Matters

In this notebook, we constructed a `SimpleScheduler` that embodies the core logic of vLLM's request scheduling. We saw how it acts like an air traffic controller, transforming a chaotic stream of incoming requests into orderly, efficient batches optimized for our resource limits.

**Key Takeaways:**
-   **The Scheduler's main job is to maximize GPU utilization.** It does this by creating full batches of requests, minimizing GPU idle time.
-   **Scheduling is a dynamic process.** As requests finish, the scheduler immediately uses the freed-up capacity to pull new requests from the waiting queue into the next batch.
-   **Scheduling is deeply linked to memory management.** In a real system like vLLM, decisions are not based on abstract token counts, but on the availability of physical memory blocks in the KV cache.

**What's Next?**
We've seen how the `EngineCore` orchestrates work and how the `Scheduler` creates batches. But who actually *runs* the model on the GPU? In the next notebook, `Model Execution: The Executor`, we will explore the component that takes these batches and performs the actual inference.